In [ ]:
from lsst.summit.utils import ConsDbClient

In [ ]:
import numpy as np
from astropy.table import Table, join
from astropy.time import Time, TimeDelta

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
%matplotlib widget

import pandas as pd

from lsst.meas.algorithms.installGaussianPsf import FwhmPerSigma

from tqdm.notebook import tqdm

import os
import pandas as pd
from matplotlib.cm import get_cmap

from lsst.ts.xml.tables.m1m3 import *
from lsst.ts.m1m3.utils import *
from lsst_efd_client import EfdClient

In [ ]:
os.environ["no_proxy"] += ",.consdb"

In [ ]:
url="http://consdb-pq.consdb:8080/consdb"

In [ ]:
consdb=ConsDbClient(url)

In [ ]:
def fetch():
    visits_query = f'''
    SELECT 
    ccdvisit1_quicklook.psf_sigma,
    ccdvisit1_quicklook.psf_ixx,
    ccdvisit1_quicklook.psf_ixy,
    ccdvisit1_quicklook.psf_iyy,
    ccdvisit1_quicklook.z4,
    ccdvisit1_quicklook.z5,
    ccdvisit1_quicklook.z6,
    ccdvisit1_quicklook.z7,
    ccdvisit1_quicklook.z8,
    ccdvisit1_quicklook.z9,
    ccdvisit1_quicklook.z10,
    ccdvisit1_quicklook.z11,
    ccdvisit1_quicklook.z12,
    ccdvisit1_quicklook.z13,
    ccdvisit1_quicklook.z14,
    ccdvisit1_quicklook.z15,
    ccdvisit1_quicklook.z16,
    ccdvisit1_quicklook.z17,
    ccdvisit1_quicklook.z18,
    ccdvisit1_quicklook.z19,
    ccdvisit1_quicklook.z20,
    ccdvisit1_quicklook.z21,
    ccdvisit1_quicklook.z22,
    ccdvisit1_quicklook.z23,
    ccdvisit1_quicklook.z24,
    ccdvisit1_quicklook.z25,
    ccdvisit1_quicklook.z26,
    ccdvisit1_quicklook.z27,
    ccdvisit1_quicklook.z28,
    ccdvisit1.detector,
    visit1.visit_id,
    visit1.seq_num,
    visit1.band,
    visit1.physical_filter,
    visit1.day_obs,
    visit1.target_name,
    visit1.airmass,
    visit1.altitude,
    visit1.azimuth,
    visit1.sky_rotation,
    visit1.exp_midpt_mjd,
    visit1.dimm_seeing,
    visit1.s_ra,
    visit1.s_dec,
    visit1_quicklook.psf_sigma_min,
    visit1_quicklook.psf_sigma_median,
    visit1_quicklook.aos_fwhm,
    visit1_quicklook.donut_blur_fwhm,
    visit1_quicklook.ringss_seeing,
    visit1_quicklook.seeing_zenith_500nm_min,
    visit1_quicklook.seeing_zenith_500nm_median,
    visit1_quicklook.physical_rotator_angle
    FROM
    cdb_lsstcam.ccdvisit1_quicklook AS ccdvisit1_quicklook,
    cdb_lsstcam.ccdvisit1 AS ccdvisit1,
    cdb_lsstcam.visit1_quicklook AS visit1_quicklook,
    cdb_lsstcam.visit1 AS visit1
    WHERE 
    ccdvisit1.ccdvisit_id = ccdvisit1_quicklook.ccdvisit_id
    AND ccdvisit1.visit_id = visit1.visit_id 
    AND visit1.visit_id = visit1_quicklook.visit_id
    AND ccdvisit1.detector NOT IN (168, 188, 123, 27, 0, 20, 65, 161)
    AND visit1.day_obs >= {20251020}
    AND visit1.science_program in ('BLOCK-T641');
    '''
    
    ccdvisits = consdb.query(visits_query).to_pandas()

    pixel_scale = 0.2
    sig2fwhm = 2 * np.sqrt(2 * np.log(2))
    ccdvisits["psf_fwhm"] = ccdvisits["psf_sigma"] * sig2fwhm * pixel_scale
    ccdvisits["psf_fwhm"] = pd.to_numeric(ccdvisits["psf_fwhm"], errors="coerce")
    
    #airmass_seeing_correction = np.array([getAirmassSeeingCorrection(airmass) for airmass in ccdvisits["airmass"]])
    #bandpass_seeing_correction = np.array([getBandpassSeeingCorrection(band) for band in ccdvisits["physical_filter"]])
    #ccdvisits["psf_fwhm_zenith_500nm"] = ccdvisits["psf_fwhm"] * airmass_seeing_correction * bandpass_seeing_correction

    ccdvisits["psf_e1"] = (ccdvisits["psf_ixx"] - ccdvisits["psf_iyy"]) / (ccdvisits["psf_ixx"] + ccdvisits["psf_iyy"])
    ccdvisits["psf_e2"] = (2. * ccdvisits["psf_ixy"]) / (ccdvisits["psf_ixx"] + ccdvisits["psf_iyy"])
    ccdvisits["psf_e"] = np.sqrt(np.array(ccdvisits["psf_e1"]**2 + ccdvisits["psf_e2"]**2, dtype=float))

    return ccdvisits

In [ ]:
ccdvisits=fetch()

In [ ]:
len(ccdvisits)

In [ ]:
zernike_cols = [f"z{i}" for i in range(4, 29)]
for z in zernike_cols:
    ccdvisits[z] = pd.to_numeric(ccdvisits[z], errors="coerce")

In [ ]:
groups = ccdvisits.groupby('visit_id')
visits_summary = pd.DataFrame({
    'day_obs': groups['day_obs'].first(),
    'seq_num': groups['seq_num'].median(),
    'donut_blur_fwhm': groups['donut_blur_fwhm'].median(),
    'aos_fwhm': groups['aos_fwhm'].median(),
    'physical_rotator_angle': groups['physical_rotator_angle'].median(),
    'altitude': groups['altitude'].median(),
    'psf_fwhm_05': groups['psf_fwhm'].quantile(0.05),
    'psf_fwhm_50': groups['psf_fwhm'].quantile(0.50),
    'psf_fwhm_95': groups['psf_fwhm'].quantile(0.95),
    'psf_e_05': groups['psf_e'].quantile(0.05),
    'psf_e_50': groups['psf_e'].quantile(0.50),
    'psf_e_95': groups['psf_e'].quantile(0.95),
    'band': groups['band'].first(),
})
visits_summary['psf_fwhm_95_05'] = np.sqrt(visits_summary['psf_fwhm_95']**2 - visits_summary['psf_fwhm_05']**2)

# Add the Zernike medians
for z in zernike_cols:
    visits_summary[z] = groups[z].median()

In [ ]:
len(visits_summary)

In [ ]:
visits_summary = visits_summary.reset_index().rename(columns={"index": "visit_id"})

In [ ]:
# Query both consDB tables
exposure = consdb.query("SELECT * FROM cdb_lsstcam.exposure WHERE science_program in ('BLOCK-T641') AND day_obs > 20251020")

visits_ql = consdb.query("SELECT * FROM cdb_lsstcam.visit1_quicklook")

# Join using astropy's join function on 'visit_id'
exposure_join = exposure.rename_column("exposure_id", "visit_id")
merged_visits = join(exposure, visits_ql, keys="visit_id", join_type="inner")  
merged_visits = join(merged_visits, Table.from_pandas(visits_summary), keys="visit_id", join_type="inner")  

In [ ]:
def fetch_efd(visit_ids):
    # visit_ids: list/array/Series of ints
    ids = sorted({int(v) for v in visit_ids})
    in_array = "ARRAY[" + ",".join(map(str, ids)) + "]"
    
    efd_query = f"""
    SELECT *
    FROM efd_lsstcam.visit1_efd AS visit1_efd
    WHERE visit1_efd.visit_id = ANY({in_array})
    """
    return consdb.query(efd_query).to_pandas()


In [ ]:
efd_visits=fetch_efd(merged_visits["visit_id"])

In [ ]:
efd_clean = efd_visits.copy()
efd_clean = efd_visits.drop(columns=["day_obs", "seq_num"], errors="ignore")

merged_visits = join(merged_visits, Table.from_pandas(efd_clean), keys="visit_id", join_type="inner")  

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 1. Convert input table to pandas DataFrame
# ============================================================
try:
    df = merged_visits.to_pandas()
except Exception:
    df = merged_visits.copy()

# ============================================================
# 2. Basic time columns and sorting
# ============================================================
df['night'] = pd.to_datetime(df['day_obs_1'].astype(str), format='%Y%m%d')
df['obs_start_dt'] = pd.to_datetime(df['obs_start'], errors='coerce', utc=True)
df = df.sort_values(['night', 'obs_start_dt'])
df = df.dropna(subset=['obs_start_dt'])  

# ============================================================
# 3. Assign test_index based on time gaps OR observation_reason change
# ============================================================
gap_seconds = 5 * 60  # 5 minutes threshold

az_jump_deg = 10.0            # new-test threshold in degrees

def assign_test_index(group):
    group = group.copy().sort_values("obs_start_dt")

    # --- Time gaps ---
    gaps = group["obs_start_dt"].diff().dt.total_seconds()

    # --- observation_reason changes ---
    reason = group["observation_reason"].astype("string")
    reason_change = reason.ne(reason.shift(1))

    # --- azimuth jumps (> 10 deg) ---
    # coerce azimuth to numeric and compute a wrapped angular difference
    az = pd.to_numeric(group["azimuth"], errors="coerce")

    # minimal angular separation between consecutive az values (handles wrap at 0/360)
    d_az = (az - az.shift(1)).abs()
    d_az = np.minimum(d_az, 360.0 - d_az)   # wrap-safe
    az_jump = d_az > az_jump_deg

    # --- New test boundary if ANY condition is met ---
    new_test = (gaps > gap_seconds) | reason_change | az_jump
    new_test = new_test.fillna(True)  # first row starts a new test

    group["test_index"] = new_test.cumsum() - 1  # start at 0
    return group

df = df.groupby("night", group_keys=False).apply(assign_test_index)

df["test_id"] = df["night"].dt.strftime("%Y-%m-%d") + "_T" + df["test_index"].astype(str)

# ============================================================
# 4. Test start/end time + duration
# ============================================================
test_times = (
    df.groupby(["night", "test_index"])["obs_start_dt"]
      .agg(test_start_time="min", test_end_time="max")
      .reset_index()
)

test_times["test_duration"] = (
    (test_times["test_end_time"] - test_times["test_start_time"])
    .dt.total_seconds() / 60.0
)

# ============================================================
# 5. Aggregation: MEAN ONLY
# ============================================================
# --- NEW COLUMN: cleaned + floored physical rotator angle ---
df['physical_rotator_angle_1'] = pd.to_numeric(df['physical_rotator_angle_1'], errors='coerce')

df.loc[df['physical_rotator_angle_1'].abs() < 3, 'physical_rotator_angle_1'] = 0
df['physical_rotator_angle_floor'] = np.floor(df['physical_rotator_angle_1'])

# --- MEAN-ONLY aggregation block (updated) ---
agg_dict = {
    'altitude_1':     'mean',
    'azimuth':      'mean',
    'physical_rotator_angle_floor': 'mean',
    'wind_speed':   'mean',
    'wind_dir':     'mean',
    'dimm_seeing':  'mean',
}

test_stats = (
    df.groupby(['night', 'test_index'])
      .agg(agg_dict)
)

test_stats = test_stats.round(0)
test_stats.columns = [f"{col}_mean" for col in test_stats.columns]
test_stats = test_stats.reset_index()

# ============================================================
# 6. Add science_program per test
# (use most frequent program inside that test)
# ============================================================
science_program_per_test = (
    df.groupby(['night', 'test_index'])['science_program']
      .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else None)
      .rename('science_program')
)

test_stats = test_stats.merge(
    science_program_per_test.reset_index(),
    on=['night', 'test_index'],
    how='left'
)

# ============================================================
# 7. Add test_start_time, test_end_time, test_duration
# ============================================================
test_stats = test_stats.merge(
    test_times,
    on=["night", "test_index"],
    how="left",
)

# Add readable columns
test_stats['night_str'] = test_stats['night'].dt.strftime('%Y-%m-%d')
test_stats['test_id'] = (
    test_stats['night_str'] + '_T' + test_stats['test_index'].astype(str)
)

cols_front = [
    "night_str", "test_start_time", "test_end_time", "test_duration",
    "science_program",
]
other_cols = [c for c in test_stats.columns if c not in cols_front and c not in ['night']]

test_stats = test_stats[cols_front + other_cols]

# ------------------------------------------------------------
# Rename output columns to human-friendly labels
# ------------------------------------------------------------
rename_map = {
    'night_str': 'Day Obs',
    'altitude_1_mean': 'Elevation',
    'azimuth_mean': 'Azimuth',
    'physical_rotator_angle_floor_mean': 'Rotator',
    'wind_speed_mean': 'Wind Spd',
    'wind_dir_mean': 'Wind Dir',
    'dimm_seeing': 'DIMM Seeing'
}

test_stats = test_stats.rename(columns=rename_map)

# ============================================================
# Count how many times each (Elevation, Azimuth, Rotator) combo occurred
# ============================================================
combo_counts = (
    test_stats
    .groupby(['Elevation', 'Azimuth', 'Rotator'])
    .size()
    .reset_index(name='Count')
)

# ============================================================
# 8. View clean, human-readable output
# ============================================================
test_stats

In [ ]:
combo_counts

In [ ]:
for col in df.columns:
    print(col)

In [ ]:
type(test_stats)

In [ ]:
df["exp_midpt"]

In [ ]:
start_time = Time(test_stats["test_start_time"].iloc[0])

***
Query and add M1M3 Thermal Gradients. 

In [ ]:
i = 0
gradient_dfs = []
efd = EfdClient('usdf_efd')

for time in test_stats["test_start_time"]:
    print(f"Querying m1m3 gradients for test starting at {time}")
    time_start = Time(time) + TimeDelta(-0.15 * 3600, format="sec")
    time_end = time_start + TimeDelta(1.0 * 3600, format="sec")
    thermocouples = ThermocoupleAnalysis(efd)
    await thermocouples.load(time_start, time_end, time_bin=30)
    gradient_dfs.append(thermocouples.xyz_r_gradients)

m1m3_gradients = pd.concat(gradient_dfs)

In [ ]:
m1m3_gradients

In [ ]:
def left_align_df_to_stacked_nearest(
    df: pd.DataFrame,
    stacked: pd.DataFrame,
    *,
    exp_col: str = "exp_midpt",
    stacked_cols: list[str] | None = None,
    prefix: str | None = None,
    tolerance: str | pd.Timedelta | None = None,
    direction: str = "nearest",
) -> pd.DataFrame:
    left = df.copy()
    left["_exp_dt_"] = pd.to_datetime(left[exp_col], errors="coerce")

    right = stacked.copy()

    # Expose stacked.index
    right = right.sort_index().reset_index()
    right = right.rename(columns={right.columns[0]: "_stack_idx_"})

    if stacked_cols is not None:
        right = right[["_stack_idx_"] + stacked_cols]

    # Timezone alignment
    ltz = getattr(left["_exp_dt_"].dt, "tz", None)
    rtz = getattr(right["_stack_idx_"].dt, "tz", None)
    if ltz is not None and rtz is None:
        right["_stack_idx_"] = right["_stack_idx_"].dt.tz_localize(ltz)
    elif ltz is None and rtz is not None:
        right["_stack_idx_"] = right["_stack_idx_"].dt.tz_convert(None)

    left = left.sort_values("_exp_dt_")
    right = right.sort_values("_stack_idx_")

    out = pd.merge_asof(
        left,
        right,
        left_on="_exp_dt_",
        right_on="_stack_idx_",
        direction=direction,
        tolerance=pd.Timedelta(tolerance) if isinstance(tolerance, str) else tolerance,
    )

    out = out.drop(columns=["_exp_dt_"])

    if prefix:
        out = out.rename(columns={c: f"{prefix}__{c}" for c in out.columns if c != "_stack_idx_"})

    return out

In [ ]:
df_aligned = left_align_df_to_stacked_nearest(
    df,
    m1m3_gradients,
    exp_col="exp_midpt",
    tolerance="60s",      # strongly recommended
    direction="nearest"
)

In [ ]:
df_aligned.columns

***
Now let's augment this with some additional EFD topics using a similar strategy. We'll start with some helper functions.

In [ ]:
def _default_colname(topic: str, field: str, index: int | None) -> str:
    # lsst.sal.ESS.temperature + temperatureItem0 + index 301 -> temperature__temperatureItem0__index301
    topic_short = topic.split(".")[-1]
    if index is None:
        return f"{topic_short}__{field}"
    return f"{topic_short}__{field}__index{index}"

import pandas as pd
from astropy.time import Time, TimeDelta
from lsst_efd_client import EfdClient

async def stack_efd_timeseries_by_tests(
    test_start_times,
    *,
    efd_client: EfdClient,
    topic: str,
    fields: list[str] | str,
    index: int | None = None,
    pre_hours: float = 0.15,
    window_hours: float = 1.0,
    time_colname: str | None = None,
    colname_map: dict[str, str] | None = None,
    colname_prefix: str | None = None,
    include_test_start_col: bool = False,
    downsample: str | None = "30s",
    on_empty: str = "skip",  # "skip" | "raise"
) -> pd.DataFrame:
    if isinstance(fields, str):
        fields = [fields]

    chunks: list[pd.DataFrame] = []
    for t0 in test_start_times:
        time_start = Time(t0) + TimeDelta(-pre_hours * 3600, format="sec")
        time_end   = time_start + TimeDelta(window_hours * 3600, format="sec")

        print(f"Querying {topic} {fields} index={index} for test starting at {t0}")

        if index is None:
            df = await efd_client.select_time_series(topic, fields, time_start, time_end)
        else:
            df = await efd_client.select_time_series(topic, fields, time_start, time_end, index=index)

        # ---- NEW: handle empty results up front ----
        if df is None or df.empty:
            msg = f"Empty result for topic={topic} index={index} window={time_start.isot}..{time_end.isot}"
            if on_empty == "raise":
                raise ValueError(msg)
            else:
                print("  ->", msg, "(skipping)")
                continue

        # ---- Ensure DatetimeIndex ----
        if not isinstance(df.index, pd.DatetimeIndex):
            cols_lower = {str(c).lower(): c for c in df.columns}
            preferred = ["time", "timestamp", "datetime", "date", "utc"]
            tcol = next((cols_lower[k] for k in preferred if k in cols_lower), None)

            # fallback: first column looks datetime-like
            if tcol is None and len(df.columns) > 0:
                maybe = pd.to_datetime(df.iloc[:, 0], errors="coerce", utc=True)
                if maybe.notna().any():
                    tcol = df.columns[0]
                    df = df.copy()
                    df[tcol] = maybe

            if tcol is None:
                raise TypeError(
                    f"EFD returned {type(df.index).__name__} for topic={topic} "
                    f"and no obvious timestamp column. Columns start: {list(df.columns)[:10]}"
                )

            df = df.copy()
            df[tcol] = pd.to_datetime(df[tcol], errors="coerce", utc=True)
            df = df.set_index(tcol)

        df = df.sort_index()
        df = df[~df.index.duplicated(keep="last")]

        # Rename
        rename: dict[str, str] = {}
        for f in fields:
            if colname_map and f in colname_map:
                out = colname_map[f]
            else:
                out = _default_colname(topic, f, index)
            if colname_prefix:
                out = f"{colname_prefix}__{out}"
            rename[f] = out
        df = df.rename(columns=rename)

        if include_test_start_col:
            df["test_start_time"] = pd.Timestamp(t0)

        chunks.append(df)

    if not chunks:
        # Return an empty df with the expected columns if you want, otherwise just empty
        return pd.DataFrame()

    stacked = pd.concat(chunks).sort_index()

    if time_colname is not None:
        stacked.index.name = time_colname

    if downsample is not None:
        stacked = stacked.resample(downsample).mean()

    return stacked

And here we specify all of the extra topics we want to get. 

In [ ]:
efd = EfdClient("usdf_efd")

m1m3_zposition = await stack_efd_timeseries_by_tests(
    test_stats["test_start_time"],
    efd_client=efd,
    topic="lsst.sal.MTM1M3.imsData",
    fields=["zPosition"],
    colname_map={
        "zPosition": "m1m3_ims_zPosition"
    },
)

m1m3_xposition = await stack_efd_timeseries_by_tests(
    test_stats["test_start_time"],
    efd_client=efd,
    topic="lsst.sal.MTM1M3.imsData",
    fields=["xPosition"],
    colname_map={
        "xPosition": "m1m3_ims_xPosition"
    },
)

m1m3_yposition = await stack_efd_timeseries_by_tests(
    test_stats["test_start_time"],
    efd_client=efd,
    topic="lsst.sal.MTM1M3.imsData",
    fields=["yPosition"],
    colname_map={
        "yPosition": "m1m3_ims_yPosition"
    },
)

tma_truss_xy = await stack_efd_timeseries_by_tests(
    test_stats["test_start_time"],
    efd_client=efd,
    topic="lsst.sal.ESS.temperature",
    index=122,
    fields=["temperatureItem6"],
    colname_map={
        "temperatureItem6": "tma_truss_xy"
    },
)

tma_truss_minusxy = await stack_efd_timeseries_by_tests(
    test_stats["test_start_time"],
    efd_client=efd,
    topic="lsst.sal.ESS.temperature",
    index=122,
    fields=["temperatureItem7"],
    colname_map={
        "temperatureItem7": "tma_truss_minus_xy"
    },
)


One more convenience function....

In [ ]:
def left_align_df_to_stacked_nearest(
    df: pd.DataFrame,
    stacked: pd.DataFrame,
    *,
    exp_col: str = "exp_midpt",
    stacked_cols: list[str] | None = None,
    prefix: str | None = None,
    tolerance: str | pd.Timedelta | None = None,
    direction: str = "nearest",
    keep_matched_time: bool = False,
) -> pd.DataFrame:
    """
    Return ONLY columns from `stacked` aligned to df[exp_col] by nearest time.

    Output index matches df.index, so you can safely do: df.join(result)
    """
    # Left: only the join key, keep df index around for restoring order
    left = df[[exp_col]].copy()
    left["_exp_dt_"] = pd.to_datetime(left[exp_col], errors="coerce")

    # Right: expose stacked.index as a column
    right = stacked.sort_index().reset_index()
    right = right.rename(columns={right.columns[0]: "_stack_time_"})

    # Choose columns to bring over
    if stacked_cols is None:
        stacked_cols = [c for c in right.columns if c != "_stack_time_"]
    right = right[["_stack_time_"] + stacked_cols]

    # Timezone alignment (common gotcha)
    ltz = getattr(left["_exp_dt_"].dt, "tz", None)
    rtz = getattr(right["_stack_time_"].dt, "tz", None)
    if ltz is not None and rtz is None:
        right["_stack_time_"] = right["_stack_time_"].dt.tz_localize(ltz)
    elif ltz is None and rtz is not None:
        right["_stack_time_"] = right["_stack_time_"].dt.tz_convert(None)

    # merge_asof requires sorted keys
    left_sorted = left.sort_values("_exp_dt_")
    right_sorted = right.sort_values("_stack_time_")

    merged = pd.merge_asof(
        left_sorted,
        right_sorted,
        left_on="_exp_dt_",
        right_on="_stack_time_",
        direction=direction,
        tolerance=pd.Timedelta(tolerance) if isinstance(tolerance, str) else tolerance,
    )

    # Restore original df row order/index
    merged = merged.reindex(left.index)

    # Return only the aligned stacked columns (and optionally matched time)
    out_cols = (["_stack_time_"] if keep_matched_time else []) + stacked_cols
    out = merged[out_cols].copy()

    if prefix:
        out = out.rename(columns={c: f"{prefix}__{c}" for c in out.columns})

    return out

Now we do all the align+stacks to add the EFD data to our dataframe

In [ ]:
df2 = df_aligned.copy()

df2 = df2.join(
    left_align_df_to_stacked_nearest(
        df,
        m1m3_xposition,
        stacked_cols=["m1m3_ims_xPosition"],
        tolerance="60s",
    )
)

df2 = df2.join(
    left_align_df_to_stacked_nearest(
        df,
        m1m3_yposition,
        stacked_cols=["m1m3_ims_yPosition"],
        tolerance="60s",
    )
)

df2 = df2.join(
    left_align_df_to_stacked_nearest(
        df,
        m1m3_zposition,
        stacked_cols=["m1m3_ims_zPosition"],
        tolerance="60s",
    )
)

df2 = df2.join(
    left_align_df_to_stacked_nearest(
        df,
        tma_truss_xy,
        stacked_cols=["tma_truss_xy"],
        tolerance="60s",
    )
)

In [ ]:
df2 = df2.join(
    left_align_df_to_stacked_nearest(
        df,
        tma_truss_minusxy,
        stacked_cols=["tma_truss_minus_xy"],
        tolerance="60s",
    )
)

In [ ]:
mtcamera_ambair = await stack_efd_timeseries_by_tests(
    test_stats["test_start_time"],
    efd_client=efd,
    topic="lsst.sal.MTCamera.utiltrunk_Body",
    fields=["ambAirtemp"],
    colname_map={
        "ambAirtemp": "mtcamera_ambAirtemp"
    },
)

mtcamera_avgtemp = await stack_efd_timeseries_by_tests(
    test_stats["test_start_time"],
    efd_client=efd,
    topic="lsst.sal.MTCamera.utiltrunk_Body",
    fields=["averageTemp"],
    colname_map={
        "averageTemp": "mtcamera_averageTemp"
    },
)

mtcamera_camBody = await stack_efd_timeseries_by_tests(
    test_stats["test_start_time"],
    efd_client=efd,
    topic="lsst.sal.MTCamera.utiltrunk_Body",
    fields=["camBodyXPlusTemp"],
    colname_map={
        "camBodyXPlusTemp": "mtcamera_camBodyXPlusTemp"
    },
)

mtcamera_domeYMinusTemp = await stack_efd_timeseries_by_tests(
    test_stats["test_start_time"],
    efd_client=efd,
    topic="lsst.sal.MTCamera.utiltrunk_Body",
    fields=["domeYMinusTemp"],
    colname_map={
        "domeYMinusTemp": "mtcamera_domeYMinusTemp"
    },
)

In [ ]:
df2 = df2.join(
    left_align_df_to_stacked_nearest(
        df,
        mtcamera_ambair,
        stacked_cols=["mtcamera_ambAirtemp"],
        tolerance="60s",
    )
)

df2 = df2.join(
    left_align_df_to_stacked_nearest(
        df,
        mtcamera_avgtemp,
        stacked_cols=["mtcamera_averageTemp"],
        tolerance="60s",
    )
)

df2 = df2.join(
    left_align_df_to_stacked_nearest(
        df,
        mtcamera_camBody,
        stacked_cols=["mtcamera_camBodyXPlusTemp"],
        tolerance="60s",
    )
)

df2 = df2.join(
    left_align_df_to_stacked_nearest(
        df,
        mtcamera_domeYMinusTemp,
        stacked_cols=["mtcamera_domeYMinusTemp"],
        tolerance="60s",
    )
)


***
Finally save the df as a pkl file to load into our analysis notebook.

In [ ]:
for col in df2.columns:
    print(col)

In [ ]:
df2.to_pickle("./BLOCK_T641_df.pkl")